# Load Address Prediction Results

This notebook reads `results.csv` and plots runtime vs. iterations for each access-pattern and core-class combination.

In [4]:
import altair as alt
import duckdb

In [5]:
con = duckdb.connect()

df = con.sql("""
    SELECT
        iters AS iterations,
        pattern,
        core_kind,
        median,
        unit
    FROM read_csv_auto('results.csv')
    ORDER BY core_kind, pattern, iterations
""").df()

df.head()

,iterations,pattern,core_kind,median,unit
0,10,random,ecore,4524,cycles
1,20,random,ecore,4567,cycles
2,30,random,ecore,4618,cycles
3,40,random,ecore,4764,cycles
4,50,random,ecore,4832,cycles


In [6]:
sa_rv_df = df[df["pattern"].isin(["strided", "sa_rv", "random"])]
unit = df["unit"].iloc[0]
y_title = f"Runtime ({unit})"
sa_rv_chart = (
    alt.Chart(sa_rv_df)
    .mark_line(point=True)
    .encode(
        x=alt.X("iterations:Q", title="Iterations"),
        y=alt.Y("median:Q", title=y_title),
        color=alt.Color("pattern:N", title="Pattern"),
        tooltip=[
            "pattern", "core_kind", "iterations", "median"
            # "min", "p25", "p75", "max",
            # "first", "last", "unit",
        ],
    )
    .properties(width=380, height=280)
    .facet(column=alt.Column("core_kind:N", title="Core Class", sort=["ecore", "pcore"]))
    .properties(title="Strided vs. SA+RV — if they match, speedup is address (not value) prediction")
    .interactive()
)

sa_rv_chart

alt.FacetChart(...)